# Теория
**Word2Vec** — это архитектура нейронных сетей и группа алгоритмов, используемых для создания векторных представлений слов (эмбеддингов).

Проще говоря, Word2Vec превращает слова в наборы чисел (векторы) таким образом, чтобы слова с похожим смыслом находились рядом в многомерном пространстве. Это позволяет компьютерам «понимать» семантические связи между словами, например, что «собака» ближе к «щенку», чем к «холодильнику».

**Основная идея**

В основе лежит дистрибутивная гипотеза: слова, которые встречаются в одинаковых контекстах, имеют схожие значения. Алгоритм обучается на больших текстах, анализируя «соседство» слов в окне фиксированного размера (например, 5 слов слева и справа от центрального).

**Как это работает (две основные модели)**

Word2Vec использует простую нейросеть с одним скрытым слоем для решения одной из двух задач:
1. CBOW (Continuous Bag of Words): модель пытается предсказать целевое слово, основываясь на его окружении (контексте). Это работает быстрее на больших данных и лучше подходит для часто встречающихся слов.

2. Skip-Gram: модель берет одно слово и пытается предсказать, какие слова могут стоять рядом с ним. Этот метод эффективнее работает с редкими словами и небольшими объемами данных.

**Ключевые особенности**

* Семантическая арифметика: обученная модель позволяет выполнять математические операции со словами.

 Классический пример: _вектор(Король) - вектор(Мужчина) + вектор(Женщина) ≈ вектор(Королева)_

* Снижение размерности: в отличие от простых методов вроде One-Hot Encoding, где размер вектора равен количеству всех слов в словаре, Word2Vec сжимает информацию в плотные векторы фиксированной длины (обычно 100–300 чисел).

* Автоматическое обучение: алгоритму не нужна ручная разметка — он обучается сам на любых доступных текстовых корпусах (Википедия, новости и т.д.).

In [43]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence
import torch
import torch.nn as nn
import torch.optim as optim

In [44]:
files = {
    'news': 'Тексты/новостной.txt',
    'science': 'Тексты/научный.txt',
    'lyric': 'Тексты/художественный.txt'
}

In [45]:
def read_and_clean(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    text = text.lower()
    text = re.sub(r'[^а-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [46]:
def tokenize(text):
    return text.split()

In [47]:
corpora = {}
for name, path in files.items():
    text = read_and_clean(path)
    tokens = tokenize(text)
    corpora[name] = tokens
    print(f"Корпус '{name}': {len(tokens)} слов, {len(set(tokens))} уникальных слов")

Корпус 'news': 3007 слов, 1724 уникальных слов
Корпус 'science': 6013 слов, 2724 уникальных слов
Корпус 'lyric': 18439 слов, 7107 уникальных слов


# Задание 1 - учим на художественном

In [48]:
def split_into_sentences(tokens, sent_len=20):
    sentences = [tokens[i:i+sent_len] for i in range(0, len(tokens), sent_len)]
    return sentences

sentences = split_into_sentences(corpora['lyric'])

sentences: Это ваш набор данных. Обычно это список списков, где каждый внутренний список — это предложение, уже разбитое на отдельные слова (токены).


vector_size=100: Размерность вектора. Каждое слово будет представлено в виде списка из 100 чисел. Чем больше это число, тем точнее модель может передать нюансы смысла, но тем больше памяти ей нужно.


window=5: Ширина «окна» контекста. Модель будет смотреть на 5 слов до и 5 слов после целевого слова, чтобы понять его значение.


min_count=2: Порог частоты. Если слово встречается во всем тексте меньше 2 раз, модель его просто проигнорирует. Это помогает избавиться от опечаток и редкого «мусора».


~~workers=4: Количество ядер процессора, которые будут использоваться для параллельного обучения. Ускоряет процесс на многоядерных системах.~~

sg=0: Выбор архитектуры.
* 0 — используется CBOW (предсказываем слово по контексту).
* 1 — используется Skip-gram (предсказываем контекст по слову).


Итог: эта команда берет ваши тексты и создает «умный словарь», где слова с похожим смыслом имеют похожие цифровые координаты.

In [49]:
model_lyric = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=0)

In [50]:
word_check = "скривила"
if word_check in model_lyric.wv:
    similar = model_lyric.wv.most_similar(word_check, topn=5)
    print(f"\n5 слов, похожих на '{word_check}':")
    for w, score in similar:
        print(f"  {w} : {score}")


5 слов, похожих на 'скривила':
  случится : 0.6312931180000305
  повторила : 0.6261417269706726
  стоял : 0.6223037242889404
  войны : 0.6200663447380066
  боль : 0.6196912527084351


# Задание 2 (поиск аналогий)

res выглядит как [('слово', 0.85)]

In [51]:
def analogy_test(model, a, c, top_k=12):
    results = model.wv.most_similar(positive=[a, c], topn=top_k)
    return [word for word, similarity in results]

In [52]:
print("баллады + поэт ≈", analogy_test(model_lyric, "баллады", "поэт"))

баллады + поэт ≈ ['и', 'как', 'не', 'а', 'в', 'она', 'что', 'но', 'это', 'на', 'я', 'лютик']


тут мне просто хотелось понять насколько логика меня и модели совпадает, на топ-12 сходстве - сошлись!

In [53]:
def analogy(model, a, b, c):
    result = model.wv.most_similar(positive=[a, c], negative=[b], topn=1)
    return result[0][0], result[0][1]


print("баллады - поэт + лютик ≈", analogy(model_lyric, "баллады", "поэт", "лютик"))

баллады - поэт + лютик ≈ ('у', 0.9847893714904785)


In [54]:
print("эльфка - кровь + цири ≈", analogy(model_lyric, "эльфка", "кровь", "цири"))

эльфка - кровь + цири ≈ ('никто', 0.911123514175415)


# Задание 3 (кос сходство)

In [55]:
def cos(model, word1, word2):
    vec1 = model.wv[word1]
    vec2 = model.wv[word2]
    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot / (norm1 * norm2)

In [56]:
close_words = [("цири", "геральт"), ("эльф", "кровь")]
print("Близкие слова:")
for i, j in close_words:
    ans = cos(model_lyric, i, j)
    print(f"  {i} - {j}: {ans}")

Близкие слова:
  цири - геральт: 0.997610867023468
  эльф - кровь: 0.9235573410987854


In [57]:
import random

In [58]:
with_ind = list(model_lyric.wv.key_to_index.keys())
i = random.choice(with_ind)
j = random.choice(with_ind)
ans = cos(model_lyric, i, j)
print(f"Случайные слова '{i}' и '{j}': {ans}")

Случайные слова 'одежде' и 'благодарю': 0.8339490294456482


model.wv — это, пожалуй, самая важная часть обученной модели Word2Vec. Расшифровывается как Word Vectors (векторы слов).
В библиотеке gensim произошло разделение: сама модель (model) нужна для обучения, а объект model.wv содержит в себе уже готовые «смысловые» векторы и методы для работы с ними.

# Задание 4 (влияние гиперпараметров)

In [59]:
hyperparams = [
    {'vector_size': 50, 'window': 2, 'sg': 0},
    {'vector_size': 100, 'window': 5, 'sg': 0},
    {'vector_size': 300, 'window': 10, 'sg': 0}
]

In [60]:
def eval_analog(model, test_analog):
    scores = []

    for a, b, c, d in test_analog:
        res = model.wv.most_similar(positive=[a, c], negative=[b], topn=1)

        if res[0][0] == d:
                scores.append(1.0)
        else:
                scores.append(0.0)
    return np.mean(scores)

In [61]:
test_analogies = [
    ("эльфка", "кровь", "цири", "никто"),
    ("геральт", "кровь", "цири", "эльф"),
    ("геральт", "ведьмак", "трисс", "чародейка"),
    ("цири", "княжна", "геральт", "ведьмак"),
    ("лютик", "поэт", "геральт", "ведьмак"),
    ("эльф", "лес", "гном", "гора"),
    ("цири", "ребенок", "геральт", "защитник"),
    ("содден", "битва", "цири", "судьба"),
    ("конь", "плотва", "ведьмак", "геральт"),
    ("рука", "пальцы", "голос", "крик"),
    ("кровь", "эльфы", "кровь", "люди")
]

ну блин ну сложно тут придумать правильно - я хз

In [62]:
for params in hyperparams:
    model_WITH_PARAMS = Word2Vec(sentences, vector_size=params['vector_size'], window=params['window'],
                          min_count=2, sg=params['sg'])
    score = eval_analog(model_WITH_PARAMS, test_analogies)
    print(f"vector_size={params['vector_size']}, window={params['window']} -> точность аналогий: {score:.2f}")

vector_size=50, window=2 -> точность аналогий: 0.00
vector_size=100, window=5 -> точность аналогий: 0.09
vector_size=300, window=10 -> точность аналогий: 0.00


# Задание 5: сравнение моделей

In [63]:
model_cbow = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=0)
model_sg = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=1)

In [64]:
freq_word = "цири"
min_freq_word = "дверь"

In [65]:
print(f"Ближайшие к '{min_freq_word}' в CBOW:")
similar_cbow = model_cbow.wv.most_similar(min_freq_word, topn=5)
for w, s in similar_cbow:
    print(f"  {w}: {s}")

print(f"\nБлижайшие к '{freq_word}' в CBOW:")
similar_cbow = model_cbow.wv.most_similar(freq_word, topn=5)
for w, s in similar_cbow:
    print(f"  {w}: {s}")

Ближайшие к 'дверь' в CBOW:
  даже: 0.8553624749183655
  больше: 0.8550177216529846
  того: 0.854559063911438
  тут: 0.8539319634437561
  хоть: 0.8538004159927368

Ближайшие к 'цири' в CBOW:
  не: 0.9986201524734497
  и: 0.9985436201095581
  в: 0.9983687996864319
  а: 0.9983397126197815
  он: 0.9982221722602844


In [66]:
print(f"Ближайшие к '{min_freq_word}' в Skip-gram:")
similar_sg = model_sg.wv.most_similar(min_freq_word, topn=5)
for w, s in similar_sg:
    print(f"  {w}: {s}")

print(f"\nБлижайшие к '{freq_word}' в Skip-gram:")
similar_sg = model_sg.wv.most_similar(freq_word, topn=5)
for w, s in similar_sg:
    print(f"  {w}: {s}")

Ближайшие к 'дверь' в Skip-gram:
  даже: 0.9965859055519104
  вы: 0.99647057056427
  или: 0.9964350461959839
  так: 0.996433436870575
  того: 0.9963876008987427

Ближайшие к 'цири' в Skip-gram:
  со: 0.9987528324127197
  вот: 0.9987367391586304
  их: 0.9986961483955383
  чтото: 0.9986497163772583
  еще: 0.9986408352851868
